In [1]:
import pandas as pd
import glob
import os

# 1. Point to your raw data folder
folder_path = r"C:\Users\USER\Desktop\cyclistic_project\01_RAW_DATA"

# 2. Find every CSV file inside that folder
csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(os.path.basename(file))

Found 12 CSV files:
202501-divvy-tripdata.csv
202502-divvy-tripdata.csv
202503-divvy-tripdata.csv
202504-divvy-tripdata.csv
202505-divvy-tripdata.csv
202506-divvy-tripdata.csv
202507-divvy-tripdata.csv
202508-divvy-tripdata.csv
202509-divvy-tripdata.csv
202510-divvy-tripdata.csv
202511-divvy-tripdata.csv
202512-divvy-tripdata.csv


In [2]:
# A list comprehension: reads each CSV file one by one into memory
df_list = [pd.read_csv(file) for file in csv_files]

# Concatenates (stacks) all DataFrames vertically into one master table
df_2025 = pd.concat(df_list, ignore_index=True)

In [3]:
print("Done! Total full-year shape:")
print(df_2025.shape)

Done! Total full-year shape:
(5552994, 13)


In [4]:
# 1. Convert timestamps from text to datetime
print("Converting timestamps...")
df_2025['started_at'] = pd.to_datetime(df_2025['started_at'])
df_2025['ended_at'] = pd.to_datetime(df_2025['ended_at'])

# 2. Feature Engineering: calculate duration and extract time components
print("Creating features: duration, day of week, month, and hour...")
df_2025['ride_length_minutes'] = (df_2025['ended_at'] - df_2025['started_at']).dt.total_seconds() / 60
df_2025['day_of_week'] = df_2025['started_at'].dt.day_name()
df_2025['month'] = df_2025['started_at'].dt.month_name()
df_2025['hour_of_day'] = df_2025['started_at'].dt.hour

# 3. Clean: filter out rides under 1 minute and over 24 hours
print("Cleaning data...")
df_clean = df_2025[(df_2025['ride_length_minutes'] >= 1) & (df_2025['ride_length_minutes'] <= 1440)].copy()

# 4. Check the row count before and after cleaning
print("Done!")
print(f"Original full-year rows: {len(df_2025):,}")
print(f"Cleaned valid rows:      {len(df_clean):,}")
print(f"Rows removed (bad data): {len(df_2025) - len(df_clean):,}")

Converting timestamps...
Creating features: duration, day of week, month, and hour...
Cleaning data...
Done!
Original full-year rows: 5,552,994
Cleaned valid rows:      5,400,008
Rows removed (bad data): 152,986


In [5]:
# 1. Total ride count by rider type for the full year
ride_counts = df_clean['member_casual'].value_counts()
print("Total Ride Counts (2025):")
print(ride_counts)

# 2. Percentage share of each rider type
ride_percentages = df_clean['member_casual'].value_counts(normalize=True) * 100
print("\nPercentage Share:")
print(ride_percentages.round(2))

Total Ride Counts (2025):
member_casual
member    3484202
casual    1915806
Name: count, dtype: int64

Percentage Share:
member_casual
member    64.52
casual    35.48
Name: proportion, dtype: float64


In [6]:
# Compare full-year duration metrics between members and casual riders
duration_summary = df_clean.groupby('member_casual')['ride_length_minutes'].agg(['mean', 'median', 'min', 'max']).round(2)
print("Full-Year Duration Summary (in minutes):")
print(duration_summary)

Full-Year Duration Summary (in minutes):
                mean  median  min      max
member_casual                             
casual         19.91   11.89  1.0  1439.98
member         12.18    8.74  1.0  1439.90


In [7]:
# 1. Define day order so days don't sort alphabetically (Friday, Monday, etc.)
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# 2. Summarize total rides and average duration by day of week
weekday_summary = df_clean.groupby(['member_casual', 'day_of_week'])['ride_length_minutes'].agg(
    total_rides='count',
    avg_duration_minutes='mean'
).round(2).reindex(days_order, level='day_of_week')

print("Weekly Behavior Summary (Full Year 2025):")
print(weekday_summary)

Weekly Behavior Summary (Full Year 2025):
                           total_rides  avg_duration_minutes
member_casual day_of_week                                   
casual        Monday            219232                 19.70
              Tuesday           216926                 17.62
              Wednesday         212744                 16.36
              Thursday          247893                 17.45
              Friday            306463                 19.60
              Saturday          395595                 22.40
              Sunday            316953                 23.10
member        Monday            493322                 11.76
              Tuesday           552529                 11.85
              Wednesday         540212                 11.63
              Thursday          565177                 11.73
              Friday            518551                 12.14
              Saturday          439876                 13.30
              Sunday            374535     

In [8]:
# Summarize total rides by hour of the day (0 to 23) for each user type
hourly_summary = df_clean.groupby(['hour_of_day', 'member_casual'])['ride_id'].count().unstack()

# Display the hourly table
print("Hourly Ride Volume (0 to 23 hours):")
print(hourly_summary)

Hourly Ride Volume (0 to 23 hours):
member_casual  casual  member
hour_of_day                  
0               36762   31391
1               23492   19107
2               15667   11430
3                8694    7520
4                6941    8606
5               11009   33525
6               25696   99153
7               47925  196413
8               68010  252034
9               68273  163882
10              82449  141036
11             105312  165364
12             123711  186824
13             126735  181968
14             133239  184831
15             148055  232972
16             168789  328673
17             183262  375149
18             156921  290116
19             116407  200336
20              83803  138362
21              71204  107972
22              59981   78612
23              43469   48926


In [9]:
# 1. Define month order chronologically
months_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

# 2. Summarize total rides by month for both user types
monthly_summary = df_clean.groupby(['month', 'member_casual'])['ride_id'].count().unstack().reindex(months_order)

# Display the monthly breakdown
print("Monthly Ride Volume (Jan - Dec 2025):")
print(monthly_summary)

Monthly Ride Volume (Jan - Dec 2025):
member_casual  casual  member
month                        
January         23405  112331
February        27003  122097
March           82864  208458
April          105260  257921
May            175655  313996
June           278702  379523
July           308446  430394
August         323533  443130
September      254727  440954
October        214380  414094
November        94719  251924
December        27112  109380


In [10]:
import os

# 1. Define the output folder path with your exact uppercase folder name
output_folder = r"C:\Users\USER\Desktop\cyclistic_project\02_PROCESSED_DATA"

# 2. Reset index so all column names are flat and ready for Power BI
monthly_export = monthly_summary.reset_index()
hourly_export = hourly_summary.reset_index()
weekday_export = weekday_summary.reset_index()

# 3. Save these clean summary files
monthly_export.to_csv(os.path.join(output_folder, "monthly_ride_trends.csv"), index=False)
hourly_export.to_csv(os.path.join(output_folder, "hourly_ride_trends.csv"), index=False)
weekday_export.to_csv(os.path.join(output_folder, "weekday_ride_trends.csv"), index=False)

print("Success! Summary files exported to 02_PROCESSED_DATA.")

Success! Summary files exported to 02_PROCESSED_DATA.
